# Feature - Development Classification - Feature Engineering

## Importing Relevant Libraries

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import seaborn as sns
import kagglehub
import torch
import cv2
from scipy.spatial import distance
from torchvision.datasets import ImageFolder
from torchvision import transforms

## Loading and Converting Dataset

### Setting up Transformers

In [12]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
]) 

### Define folder paths

In [15]:
base_path = "../../dataset/Dataset"
train_path = f"{base_path}/Images/Emotion/train"
test_path  = f"{base_path}/Images/Emotion/test"

### Creating the datasets

In [16]:
train_dataset = ImageFolder(train_path, transform=transform)
test_dataset = ImageFolder(test_path, transform=transform)

## Feature Extraction

### Feature Extraction Logic

Features to Extract from the images


*   **Fill ratio** - (Pixels with drawing) / (Total Pixels)

tiny figures in the corner (timidity/anxiety) or fill the whole page (confidence/impulsivity)

*   **Pressure/Stroke Thickness** - Average pixel intensity of the edges.

Heavy, dark lines often indicate high energy or aggression. Faint, sketchy lines can indicate hesitation or low energy.



*   **Color Palette (Warm vs. Cool)** - Dominant Hue analysis.

Predominance of Red/Orange (Warm) vs. Blue/Green (Cool) vs. Black/White.


*   **"Blob" Count (Object Individuation)** - Number of closed contours.

Younger children draw disconnected items. Older children connect them into a scene.


*   Centroid Logic (Centrality):

Is the main subject in the center (egocentric) or spread out?

In [17]:
class DrawingAnalyser:
    def __init__(self, image_path):
        self.image_path = image_path
        self.img_color = cv2.imread(self.image_path)
        
        if self.img_color is None:
            self.valid = False
            return 
        self.valid = True
        
        # Resize for consistent processing
        h, w = self.img_color.shape[:2]
        scale = 500 / w
        dim = (500, int(h * scale))
        self.img_color = cv2.resize(self.img_color, dim, interpolation=cv2.INTER_AREA)
        self.img_gray = cv2.cvtColor(self.img_color, cv2.COLOR_BGR2GRAY)
        
        self.height, self.width = self.img_gray.shape
        self.total_pixels = self.height * self.width
        
    def _thresholding(self):
        """Thresholding"""
        _, thresh = cv2.threshold(self.img_gray, 200, 255, cv2.THRESH_BINARY_INV)
        return thresh
    
    def _space_utilization(self, thresh):
        """Space utilization"""
        non_zero = cv2.countNonZero(thresh)
        fill_ratio = non_zero / self.total_pixels
        return fill_ratio, non_zero   # <-- FIXED
    
    def _pressure(self, thresh, non_zero):
        """Average Pressure"""
        if non_zero == 0:
            return 0
        
        mask = thresh > 0
        raw_values = self.img_gray[mask]
        avg_pressure = np.mean(255 - raw_values)
        return avg_pressure
    
    def _color_palette(self):
        """Color palette"""
        hsv = cv2.cvtColor(self.img_color, cv2.COLOR_BGR2HSV)
        h, s, v = cv2.split(hsv)
        
        saturation_mask = s > 40
        valid_hues = h[saturation_mask]
        
        if len(valid_hues) == 0:
            return 0.5  # neutral / unknown 
        
        warm_count = np.sum((0 <= valid_hues) & (valid_hues <= 35) |
                            (170 <= valid_hues) & (valid_hues <= 180))
        cool_count = len(valid_hues) - warm_count
        
        total = warm_count + cool_count + 1e-5
        warm_ratio = warm_count / total
        
        return warm_ratio
    
    def _blob_count(self, thresh):
        """Blob count"""
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        return len(contours)
    
    def _centrality(self, thresh, non_zero):
        if non_zero == 0:
            return 0
        
        M = cv2.moments(thresh)
        if M["m00"] == 0:
            return 0
        
        cX = int(M["m10"] / M["m00"])
        cY = int(M["m01"] / M["m00"])
        
        image_center = (self.width // 2, self.height // 2)
        
        dist_from_center = distance.euclidean((cX, cY), image_center)
        max_dist = distance.euclidean((0, 0), image_center)
        
        return 1.0 - (dist_from_center / max_dist)
    
    
    def analyse(self):
        """Public API"""
        if not self.valid:
            return None
        
        thresh = self._thresholding()
        fill_ratio, non_zero = self._space_utilization(thresh)  # now works
        avg_pressure = self._pressure(thresh, non_zero)
        warm_ratio = self._color_palette()
        blob_count = self._blob_count(thresh)
        centrality = self._centrality(thresh, non_zero)
        
        return {
            "fill_ratio": round(fill_ratio, 4),
            "avg_pressure": round(avg_pressure, 2),
            "warm_color_ratio": round(warm_ratio, 2),
            "blob_count": blob_count,
            "centrality": round(centrality, 4),
        }


### Function to analyse drawings using the class 

In [18]:
def analyse_drawing(image_path):
    analyser = DrawingAnalyser(image_path)
    return analyser.analyse()

### Combined Excecution Loop

#### Defining datasets to process

In [19]:
data_list = []  

In [20]:
datasets_to_process = [
    ("Train", train_dataset),
    ("Test", test_dataset),
]

#### Extracting features from datasets

In [21]:
for split_name, dataset in datasets_to_process:
    print(f"--- Processing {split_name} Set ---")

    # Iterate through the samples in this specific dataset
    for i, (path, class_idx) in enumerate(dataset.samples):

        # 1. Run analysis
        features = analyse_drawing(path)

        if features:
            # 2. Add Metadata
            features["label_idx"] = class_idx
            features["dataset_source"] = split_name  # 'Train' or 'Test'
            features["filename"] = os.path.basename(path)

            data_list.append(features)

        if i % 1000 == 0:
            print(f"Processed {i} images in {split_name}...")

--- Processing Train Set ---
Processed 0 images in Train...
Processed 1000 images in Train...
Processed 2000 images in Train...
Processed 3000 images in Train...
Processed 4000 images in Train...
Processed 5000 images in Train...
Processed 6000 images in Train...
Processed 7000 images in Train...
Processed 8000 images in Train...
Processed 9000 images in Train...
--- Processing Test Set ---
Processed 0 images in Test...
Processed 1000 images in Test...


## Converting to dataframe

In [22]:
df = pd.DataFrame(data_list)

### Map class index to name 

In [23]:
idx_to_class = {v: k for k, v in train_dataset.class_to_idx.items()}
df["label_name"] = df["label_idx"].map(idx_to_class)

### Reorder columns

In [24]:
cols = ["filename", "dataset_source", "label_name", "fill_ratio", "avg_pressure", "warm_color_ratio", "blob_count", "centrality"]
df = df[cols]

### Feature engineered dataset

In [25]:
df.head()

,filename,dataset_source,label_name,fill_ratio,avg_pressure,warm_color_ratio,blob_count,centrality
0,101-1A-1240-M-H.jpg,Train,Happiness,0.1436,110.57,0.53,204,0.7977
1,101-1A-1298-M-H.jpg,Train,Happiness,0.0977,112.21,0.39,111,0.8133
2,101-1A-141-M-H.jpg,Train,Happiness,0.0551,138.85,0.50,88,0.6214
3,101-1A-1543-M-H.jpg,Train,Happiness,0.0663,106.86,0.57,85,0.9383
4,101-1A-331-F-H.jpg,Train,Happiness,0.1426,99.45,0.50,71,0.7955


In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10860 entries, 0 to 10859
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   filename          10860 non-null  object 
 1   dataset_source    10860 non-null  object 
 2   label_name        10860 non-null  object 
 3   fill_ratio        10860 non-null  float64
 4   avg_pressure      10860 non-null  float64
 5   warm_color_ratio  10860 non-null  float64
 6   blob_count        10860 non-null  int64  
 7   centrality        10860 non-null  float64
dtypes: float64(4), int64(1), object(3)
memory usage: 678.9+ KB


## Saving the dataset

In [27]:
csv_path = "../data/drawing_features.csv"
df.to_csv(csv_path, index=False)